In [6]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
import numpy as np

# Load data
df = pd.read_csv('data/claims_train.csv')

# Select features and target
X = df[['Exposure','Area','VehPower','VehAge','DrivAge','BonusMalus','VehBrand','VehGas','Density','Region']]
y = df['ClaimNb']

# One-hot encode categorical features
X_encoded = pd.get_dummies(X, drop_first=True)

# Store importances
all_importances = []

# Fit 10 trees with different random states
for i in range(10):
    tree = DecisionTreeRegressor(random_state=i)
    tree.fit(X_encoded, y)
    all_importances.append(tree.feature_importances_)

# Average importances across all trees
avg_importances = np.mean(all_importances, axis=0)

# Combine into a dataframe
feat_importance_df = pd.DataFrame({
    'Feature': X_encoded.columns,
    'AvgImportance': avg_importances
}).sort_values(by='AvgImportance', ascending=False)

print(feat_importance_df)


           Feature  AvgImportance
5          Density       0.194081
3          DrivAge       0.188182
2           VehAge       0.123747
0         Exposure       0.120999
1         VehPower       0.079239
4       BonusMalus       0.061687
21  VehGas_Regular       0.024877
16     VehBrand_B2       0.020734
38      Region_R82       0.012651
17     VehBrand_B3       0.011587
19     VehBrand_B5       0.010552
41      Region_R93       0.010380
8           Area_D       0.009675
7           Area_C       0.009023
25      Region_R24       0.008653
20     VehBrand_B6       0.007687
32      Region_R52       0.007395
18     VehBrand_B4       0.007362
33      Region_R53       0.007246
13    VehBrand_B12       0.006952
40      Region_R91       0.006774
6           Area_B       0.006611
9           Area_E       0.006114
28      Region_R31       0.005839
35      Region_R72       0.005722
34      Region_R54       0.005287
11    VehBrand_B10       0.004323
14    VehBrand_B13       0.003942
12    VehBrand

In [22]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

X_important = df[['Density', 'DrivAge', 'VehAge', 'Exposure', 'VehPower', 'BonusMalus']]
y = df['ClaimNb']

x_train, x_test, y_train, y_test = train_test_split(X_important, y, test_size=.2)
clf = DecisionTreeRegressor(random_state=42, min_samples_leaf=5)
clf.fit(x_train, y_train)
y_hat = clf.predict(x_test)

print('MSE:', mean_squared_error(y_test, y_hat))

print(y_hat[:100])


MSE: 0.07297900219445114
[0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.125      0.
 0.         0.         0.         0.         0.         0.
 0.2        0.         0.         0.         0.2        0.
 0.33333333 0.         0.         0.         0.         0.
 0.         0.         0.25       0.42857143 0.2        0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.33333333 0.
 0.         0.         0.         0.         0.         0.
 0.         0.33333333 0.71428571 0.         0.2        0.25
 0.         0.         0.         0.125      0.         0.
 0.         0.42857143 0.         0.         0.         0.2
 0.         0.2        0.         0.         0.         0.
 0.         0.         0.         0.4        0.         0.
 0.         0.         0.14285714 0.         0.         0.
 0.         0.         0.         0.         0.4        0.
 0.         0.         0.   

In [24]:
def rss(y):
    return len(y) * np.var(y) if len(y) > 0 else 0

def proportions(region):
    """Return class 0 and 1 proportions in a region"""
    if len(region) == 0:
        return (0, 0)
    count0 = sum(1 for _, ci in region if ci == 0)
    count1 = sum(1 for _, ci in region if ci == 1)
    total = count0 + count1
    return (count0 / total, count1 / total)

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

def best_split(X, y):
    n_samples, n_features = X.shape
    best_feature, best_threshold = None, None
    best_rss = float("inf")

    for feature in range(n_features):
        values = X[:, feature]
        thresholds = np.unique(values)
        for t in thresholds:
            left_idx = values < t
            right_idx = values >= t
            if np.sum(left_idx) == 0 or np.sum(right_idx) == 0:
                continue

            rss_left = rss(y[left_idx])
            rss_right = rss(y[right_idx])
            total_rss = rss_left + rss_right

            if total_rss < best_rss:
                best_rss = total_rss
                best_feature = feature
                best_threshold = t

    return best_feature, best_threshold

# --- Recursive Tree Builder ---
def build_tree(X, y, depth=0, max_depth=10, min_leaves = 5):
    
    if depth >= max_depth or len(np.unique(y)) == 1 or len(y) < min_leaves:
        return Node(value=np.mean(y))

    feature, threshold = best_split(X, y)
    if feature is None:
        return Node(value=np.mean(y))

    left_idx = X[:, feature] < threshold
    right_idx = X[:, feature] >= threshold

    left = build_tree(X[left_idx], y[left_idx], depth+1, max_depth)
    right = build_tree(X[right_idx], y[right_idx], depth+1, max_depth)
    return Node(feature, threshold, left, right)

# --- Prediction ---
def predict_one(node, x):
    if node.value is not None:
        return node.value
    if x[node.feature] < node.threshold:
        return predict_one(node.left, x)
    else:
        return predict_one(node.right, x)

def predict(node, X):
    return np.array([predict_one(node, x) for x in X])


X = df[['Density', 'DrivAge', 'VehAge', 'Exposure', 'VehPower', 'BonusMalus']].to_numpy()
y = df['ClaimNb'].to_numpy()

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tree = build_tree(x_train, y_train)
y_pred = predict(tree, x_test)

rmse = np.sqrt(np.mean((y_pred - y_test) ** 2))

print(f"RMSE: {rmse:.2f}")

RMSE: 0.24
